In [0]:
create streaming table orders_bronze 
as
select *, _metadata.file_path as file_name,current_timestamp() as load_time 
from cloud_files('/Volumes/workspace/landing/landing/orders/','csv',map('cloudFiles.inferColumnTypes','True'));


create streaming table customers_bronze
as 
select *,_metadata.file_path as file_name,current_timestamp() as load_time 
from cloud_files('/Volumes/workspace/landing/landing/customers/','csv',map('cloudFiles.inferColumnTypes','True'))

In [0]:
create streaming table orders_silver_cleaned
( constraint valid_orders expect(id is not null)
  on violation drop row
) as
select orderid as id,orderdate as order_date,totalamount as total_amount,status,file_name,load_time
from stream(live.orders_bronze) 

In [0]:
create streaming table customer_silver_cleaned(
  constraint valid_customer expect(customer_id is not null)
  on violation drop row
)
as
select customerid as customer_id,customername as customer_name,address as city,dateofbirth as dob,registrationdate as customer_since,load_time
from stream(live.customers_bronze)

In [0]:
create streaming table customers_silver;
apply changes into customers_silver
from stream(live.customer_silver_cleaned)
keys(customer_id)
sequence by load_time
stored as scd type 2;

In [0]:
create streaming table orders_silver;
apply changes into orders_silver
from stream(live.orders_silver_cleaned)
keys(id)
sequence by load_time;

In [0]:
create materialized view city_wise_sales_gold
as 
select city,sum(total_amount) as total_sales
from live.orders_silver o
join live.customers_silver c
on o.id=c.customer_id
group by city